# Notebook 2: Online Feature Store Setup

Sets up the Postgres-backed Online Service, registers stream feature views with continuous aggregation for all 4 entity velocity groups, and benchmarks freshness and lookup latency.

---

### Architecture

```
FRAUD_TRANSACTIONS (source of truth)
      │
      └──► REST Ingest API ──► Online Feature Store (Postgres)
                                  ├── Stream FVs: CONTINUOUS aggregation (< 2s)
                                  └── Batch FV: Customer profile (daily)
```

### Feature Coverage

| Entity | Stream aggregations | Notes |
|---|---|---|
| Customer | count, sum, max, min, approx_count_distinct — 1h/6h/24h/48h/1wk | Primary card-testing signal |
| Merchant | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Merchant under attack |
| Wallet DPAN | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Compromised card |
| IP Address | count, sum, approx_count_distinct — 1h/6h/24h/48h/1wk | Bot farm |
| Customer Profile | Lifetime stats, account age | Batch online FV, daily refresh |

Derived ratio features (velocity ratios, concentration, deviation) are computed inline in the SPCS scoring container.

> **Preview note:** Requires `snowflake-ml-python >= 1.41`. Auth token is derived automatically from the active session when running inside Snowflake Notebooks.

In [ ]:
# ── Package version check ──────────────────────────────────────────────────
# IMPORTANT: If you just ran !pip install, you MUST restart the kernel before
# proceeding. Session → Restart Kernel → then re-run all cells from the top.
# pip install updates packages on disk but the running kernel keeps old modules
# in memory until a full restart.
#
# To avoid this: add snowflake-ml-python via the Packages picker at the top of
# the notebook (search "snowflake-ml-python", pin to >=1.41). That rebuilds the
# environment automatically and persists across sessions.
# ───────────────────────────────────────────────────────────────────────────

import importlib.metadata, sys

v = importlib.metadata.version('snowflake-ml-python')
major, minor = int(v.split('.')[0]), int(v.split('.')[1])
if (major, minor) < (1, 41):
    raise RuntimeError(
        f'snowflake-ml-python {v} is too old. Requires >= 1.41.\n'
        'Install via the Packages picker at the top of this notebook,\n'
        'then restart the kernel and re-run from the top.'
    )

# Verify the method actually exists in the running kernel (not just on disk)
from snowflake.ml.feature_store import FeatureStore as _FS
if not hasattr(_FS, 'create_online_service'):
    raise RuntimeError(
        f'create_online_service not found on FeatureStore (sdk={v}).\n'
        'Kernel is running an old cached version.\n'
        'Restart the kernel (Session → Restart Kernel) then re-run all cells.'
    )

print(f'snowflake-ml-python {v} — OK')
print(f'create_online_service available — OK')
print(f'Python: {sys.executable}')

In [ ]:
# ── Programmatic Access Token (PAT) setup ────────────────────────────────────
# The Online Feature Store REST API requires a PAT — session tokens are rejected
# by the OFS ingress proxy regardless of whether you use the SDK or direct REST.
#
# ONE-TIME SETUP: Create a PAT in Snowsight:
#   1. Click your profile icon (top-left) → 'Programmatic Access Tokens'
#   2. Click 'Add Token'
#   3. Name: FRAUD_DEMO_PAT   Role: FRAUD_MLOPS   Expiry: 30 days
#   4. Copy the token value (shown ONCE — save it)
#   5. Paste it in the cell below and run
#
# The PAT is stored in os.environ so all subsequent OFS SDK calls find it.

import os

# ── Paste your PAT here ───────────────────────────────────────────────────────
OFS_PAT = ''   # <-- paste your PAT between the quotes
# ─────────────────────────────────────────────────────────────────────────────

if OFS_PAT:
    os.environ['SNOWFLAKE_PAT'] = OFS_PAT
    print(f'PAT set: {OFS_PAT[:16]}...  (OFS ingest + query will work)')
else:
    print('WARNING: OFS_PAT is empty.')
    print('The freshness benchmark (section 2.6) will be skipped.')
    print('Create a PAT in Snowsight Profile -> Programmatic Access Tokens')
    print('then paste it above and re-run this cell.')


In [ ]:
import time, os, numpy as np, requests, random, concurrent.futures
from datetime import datetime
from snowflake.snowpark.context import get_active_session

# Correct Preview API imports — requires snowflake-ml-python >= 1.41
from snowflake.ml.feature_store import (
    FeatureStore, FeatureView, Entity, CreationMode,
    OnlineConfig, OnlineStoreType, StreamSource, StreamConfig, Feature,
    FeatureGroup,
)
from snowflake.ml.feature_store.spec.enums import FeatureAggregationMethod
from snowflake.ml.feature_store import online_service as ofs_utils
from snowflake.ml.feature_store.online_service import OnlineServiceAccess
from snowflake.snowpark.types import (
    StructType, StructField, StringType, FloatType,
    TimestampType, TimestampTimeZone,
)

session = get_active_session()
session.sql('USE ROLE FRAUD_MLOPS').collect()
session.sql('USE DATABASE FRAUD_DEMO_DEV').collect()
session.sql('USE SCHEMA FEATURES').collect()
session.sql('USE WAREHOUSE FRAUD_DEMO_WH').collect()

fs = FeatureStore(
    session=session,
    database='FRAUD_DEMO_DEV',
    name='FEATURE_STORE',
    default_warehouse='FRAUD_DEMO_WH',
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
    # PUBLIC: nb02 runs in Snowflake Notebooks, NOT inside SPCS.
    # OnlineServiceAccess.INTERNAL forces SPCS-cluster-internal URLs which are
    # unreachable from Notebooks — all REST calls would fail with a connection error.
    # Use PUBLIC here; use INTERNAL only in nb04/nb06 which run inside the SPCS scoring container.
    online_service_access=OnlineServiceAccess.PUBLIC,
)

# Session token — no PAT env var needed when running inside Snowflake Notebooks
PAT = session.connection.rest.token or os.environ.get('SNOWFLAKE_PAT', '')
if not PAT:
    raise RuntimeError('Could not obtain session token. If running locally set SNOWFLAKE_PAT.')
print(f'Auth token     : from active session ({PAT[:12]}...)')
print('Feature Store  : FRAUD_DEMO_DEV.FEATURE_STORE')

## 2.1 Create Online Service

Provisions a Postgres-backed serving layer co-located with your Snowflake account. Created once per Feature Store — takes 3-5 minutes on first run.

Exposes REST ingest and query endpoints (public, PrivateLink, or SPCS-internal). For production, configure PrivateLink to keep traffic off the public internet.

In [ ]:
print('Creating Online Service...\nFirst creation takes 3-5 minutes. Polling until RUNNING.\n')

try:
    fs.create_online_service('FRAUD_MLOPS', 'FRAUD_MLOPS')
except Exception as e:
    if 'already exists' in str(e).lower():
        print('Online service already exists — continuing')
    else:
        raise

status = fs.get_online_service_status()
start  = time.time()
while status.status != 'RUNNING':
    print(f'  [{time.time()-start:.0f}s] {status.status} — waiting 30s...')
    time.sleep(30)
    status = fs.get_online_service_status()

print(f'Online Service RUNNING ({time.time()-start:.0f}s)')

# ── Resolve public endpoint URLs ─────────────────────────────────────────────
# SYSTEM$GET_FEATURE_STORE_ONLINE_SERVICE_STATUS returns:
#   {"endpoints": [{"name": "ingest", "url": "https://...", "internal_url": "http://...svc.spcs.internal"}]}
# We use 'url' (public HTTPS) from Notebooks — internal URLs only work from within SPCS.
import json as _json

_raw = session.sql(
    "SELECT SYSTEM$GET_FEATURE_STORE_ONLINE_SERVICE_STATUS('FRAUD_DEMO_DEV.FEATURE_STORE')"
).collect()[0][0]
_svc       = _json.loads(_raw)
_endpoints = _svc.get('endpoints', [])

def _pick_url(ep_name, prefer_internal=False):
    """Extract URL from endpoints list or dict. Handles both formats."""
    if isinstance(_endpoints, list):
        # SDK returns a list: [{"name": "ingest", "url": "...", "internal_url": "..."}]
        for ep in _endpoints:
            if ep.get('name') == ep_name:
                if prefer_internal:
                    url = ep.get('internal_url') or ep.get('url', '')
                else:
                    url = ep.get('url') or ep.get('internal_url', '')
                return url.rstrip('/')
        raise RuntimeError(f'Endpoint {ep_name!r} not found. Available: {[e.get("name") for e in _endpoints]}')
    # Dict format fallback
    ep = _endpoints.get(ep_name, _endpoints.get(ep_name.lower(), {}))
    if isinstance(ep, str): return ep.rstrip('/')
    if prefer_internal:
        url = ep.get('internal_url') or ep.get('internal') or ep.get('url', '')
    else:
        url = ep.get('url') or ep.get('public') or ep.get('internal_url', '')
    if not url:
        raise RuntimeError(f'No URL for {ep_name!r}. Available: {json.dumps(_endpoints, indent=2)}')
    return url.rstrip('/')

QUERY_URL  = _pick_url('query')
INGEST_URL = _pick_url('ingest')
HEADERS    = {'Authorization': f'Snowflake Token="{PAT}"', 'Content-Type': 'application/json'}

print(f'  QUERY_URL  : {QUERY_URL}')
print(f'  INGEST_URL : {INGEST_URL}')
print(f'  Auth token : {PAT[:16]}...')


## 2.2 Register Entities

In [ ]:
# An Entity is a named primary key definition — it tells the Feature Store which column(s)
# to use when joining feature views to incoming scoring requests.
# join_keys must match the column names in both the stream schema and the scoring payload.
# We define 4 entities (one per velocity dimension): Customer, Merchant, DPAN, IP Address.
# Card Token velocity is derived at scoring time from DPAN features, so no separate entity needed.
customer_entity = Entity(name='FRAUD_CUSTOMER',    join_keys=['CUSTOMER_ID'])
merchant_entity = Entity(name='FRAUD_MERCHANT',    join_keys=['MERCHANT_ID'])
dpan_entity     = Entity(name='FRAUD_WALLET_DPAN', join_keys=['WALLET_DPAN'])
ip_entity       = Entity(name='FRAUD_IP_ADDRESS',  join_keys=['IP_ADDRESS'])

# Register each entity in the Feature Store (idempotent — safe to re-run).
# Entities are lightweight metadata objects; they don't store any data themselves.
for entity in [customer_entity, merchant_entity, dpan_entity, ip_entity]:
    try:
        fs.register_entity(entity)
        print(f'  Registered: {entity.name}')
    except Exception as e:
        if 'already exists' in str(e).lower():
            print(f'  Exists: {entity.name}')
        else:
            raise

## 2.3 Register Stream Source

Defines the event schema for real-time transaction ingestion. Every transaction is sent here via REST to trigger continuous velocity feature updates.

In [ ]:
from snowflake.snowpark.types import DoubleType

# A StreamSource defines the schema of the event records that will be ingested into the
# Online Feature Store. Each time a transaction is scored, a record is posted to the
# INGEST endpoint — the OFS uses these records to maintain rolling window aggregates.
#
# Only the fields needed for feature computation are included here (not the full transaction).
# AMOUNT_USD: the transaction amount — used for sum/count/max velocity features.
# IS_GBR:     1.0 if merchant country = GBR, else 0.0 — precomputed to avoid string ops.
# EVENT_TS:   the transaction timestamp — defines the time axis for all rolling windows.
#             TimestampTimeZone.NTZ = no time zone (UTC assumed, consistent with TIMESTAMP_NTZ).
txn_stream = StreamSource(
    name='FRAUD_TXN_EVENTS',
    schema=StructType([
        StructField('CUSTOMER_ID', StringType()),
        StructField('MERCHANT_ID', StringType()),
        StructField('WALLET_DPAN', StringType()),
        StructField('IP_ADDRESS',  StringType()),
        StructField('AMOUNT_USD',  DoubleType()),
        StructField('IS_GBR',      DoubleType()),
        StructField('EVENT_TS',    TimestampType(TimestampTimeZone.NTZ)),
    ]),
    desc='Real-time fraud transaction events',
)

# register_stream_source creates the underlying Snowflake stream object.
# Safe to re-run — catches 'already exists' errors gracefully.
try:
    fs.register_stream_source(txn_stream)
    print('Stream source registered: FRAUD_TXN_EVENTS')
except Exception as e:
    if 'already exists' in str(e).lower():
        print('Stream source already exists — continuing')
    else:
        raise

## 2.4 Create Stream Feature Views (CONTINUOUS Aggregation)

`FeatureAggregationMethod.CONTINUOUS` maintains running aggregates updated on every ingest event — no refresh cycle, no warehouse, < 2s end-to-end freshness.

**Why this matters:** A card-testing burst of 5 transactions spanning 30 seconds is invisible to a 33-39s refresh cycle. With CONTINUOUS aggregation, velocity counts update after each transaction — the model detects the burst from transaction 2 onwards.

> **approx_count_distinct note:** Distinct-value features (distinct merchants, DPANs) use HyperLogLog (~6.5% RSE at default precision). For fraud velocity signals this is negligible. Use `precision=14` to reduce RSE to ~0.8% if needed.

In [ ]:
import pandas as pd
from datetime import datetime

online_cfg = OnlineConfig(enable=True, store_type=OnlineStoreType.POSTGRES)

def identity_transform(df: pd.DataFrame) -> pd.DataFrame:
    """Pass events through unchanged — all aggregations handled by Feature.count/sum/max."""
    return df

backfill_schema = StructType([
    StructField('CUSTOMER_ID', StringType()),
    StructField('MERCHANT_ID', StringType()),
    StructField('WALLET_DPAN', StringType()),
    StructField('IP_ADDRESS',  StringType()),
    StructField('AMOUNT_USD',  DoubleType()),
    StructField('IS_GBR',      DoubleType()),
    StructField('EVENT_TS',    TimestampType(TimestampTimeZone.NTZ)),
])
backfill_df = session.create_dataframe(
    [['DUMMY', 'DUMMY', 'DUMMY', '0.0.0.0', 0.0, 0.0, datetime(2020, 1, 1)]],
    schema=backfill_schema,
)

stream_cfg = StreamConfig(
    stream_source=txn_stream,
    transformation_fn=identity_transform,
    backfill_df=backfill_df,
)

# ── Customer velocity (primary card-testing signal) ──────────────────────────
# CONTINUOUS: features update on every REST ingest event, not on a batch schedule.
# refresh_freq / feature_granularity: '1 minute' tile size for the online store.
customer_fv = FeatureView(
    name='FRAUD_CUSTOMER_VELOCITY',
    entities=[customer_entity],
    stream_config=stream_cfg,
    timestamp_col='EVENT_TS',
    refresh_freq='1 minute',
    feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('PURCHASES_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('PURCHASES_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('PURCHASES_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('PURCHASES_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('PURCHASES_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '1h' ).alias('PURCHASES_AMT_L1H'),
        Feature.sum(   'AMOUNT_USD',   '6h' ).alias('PURCHASES_AMT_L6H'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('PURCHASES_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '48h').alias('PURCHASES_AMT_L48H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('PURCHASES_AMT_L1WK'),
        Feature.max(   'AMOUNT_USD',   '1h' ).alias('PURCHASES_MAX_L1H'),
        Feature.max(   'AMOUNT_USD',   '24h').alias('PURCHASES_MAX_L24H'),
        Feature.max(   'AMOUNT_USD',   '7d' ).alias('PURCHASES_MAX_L1WK'),
        Feature.min(   'AMOUNT_USD',   '24h').alias('PURCHASES_MIN_L24H'),
        Feature.approx_count_distinct('MERCHANT_ID', '1h' ).alias('DISTINCT_MERCHANTS_L1H'),
        Feature.approx_count_distinct('MERCHANT_ID', '6h' ).alias('DISTINCT_MERCHANTS_L6H'),
        Feature.approx_count_distinct('MERCHANT_ID', '24h').alias('DISTINCT_MERCHANTS_L24H'),
        Feature.approx_count_distinct('MERCHANT_ID', '7d' ).alias('DISTINCT_MERCHANTS_L1WK'),
        Feature.approx_count_distinct('WALLET_DPAN', '24h').alias('DISTINCT_DPANS_L24H'),
        Feature.approx_count_distinct('WALLET_DPAN', '7d' ).alias('DISTINCT_DPANS_L1WK'),
        Feature.count('IS_GBR', '24h').alias('PURCHASES_GBR_NUM_L24H'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='Customer velocity — CONTINUOUS, < 2s freshness',
)
try:
    customer_fv = fs.register_feature_view(customer_fv, version='V1', block=True)
    print('  Registered: FRAUD_CUSTOMER_VELOCITY')
except Exception as e:
    if 'already exists' in str(e).lower():
        customer_fv = fs.get_feature_view('FRAUD_CUSTOMER_VELOCITY', 'V1')
        print('  Already registered: FRAUD_CUSTOMER_VELOCITY')
    else:
        raise

In [ ]:
# ── Merchant velocity ────────────────────────────────────────────────────────
# Merchant-level features detect compromised merchants and bust-out fraud rings.
# Key signal: a sudden spike in MERCHANT_UNIQUE_CUST_L1H (many different customers
# at one merchant in 1 hour) can indicate a card-skimming or POS-compromise incident.
merchant_fv = FeatureView(
    name='FRAUD_MERCHANT_VELOCITY',
    entities=[merchant_entity],
    stream_config=stream_cfg, timestamp_col='EVENT_TS',
    refresh_freq='1 minute', feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('MERCHANT_TXN_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('MERCHANT_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('MERCHANT_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('MERCHANT_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('MERCHANT_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '1h' ).alias('MERCHANT_TXN_AMT_L1H'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('MERCHANT_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('MERCHANT_TXN_AMT_L1WK'),
        Feature.max(   'AMOUNT_USD',   '24h').alias('MERCHANT_MAX_TXN_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '1h' ).alias('MERCHANT_UNIQUE_CUST_L1H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('MERCHANT_UNIQUE_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '7d' ).alias('MERCHANT_UNIQUE_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='Merchant velocity — CONTINUOUS',
)
try:
    merchant_fv = fs.register_feature_view(merchant_fv, version='V1', block=True)
    print('  Registered: FRAUD_MERCHANT_VELOCITY')
except Exception as e:
    if 'already exists' in str(e).lower():
        merchant_fv = fs.get_feature_view('FRAUD_MERCHANT_VELOCITY', 'V1')
        print('  Already registered: FRAUD_MERCHANT_VELOCITY')
    else:
        raise

# ── Wallet DPAN velocity ──────────────────────────────────────────────────────
# DPAN = Device Primary Account Number (the tokenised card number provisioned to a wallet).
# Key signal: DPAN_DISTINCT_CUST_L24H > 1 means the same device token is being used by
# multiple customers — a strong indicator of DPAN cloning or account sharing fraud.
dpan_fv = FeatureView(
    name='FRAUD_DPAN_VELOCITY',
    entities=[dpan_entity],
    stream_config=stream_cfg, timestamp_col='EVENT_TS',
    refresh_freq='1 minute', feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('DPAN_TXN_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('DPAN_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('DPAN_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('DPAN_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('DPAN_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('DPAN_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('DPAN_TXN_AMT_L1WK'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('DPAN_DISTINCT_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '7d' ).alias('DPAN_DISTINCT_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='DPAN velocity — shared DPAN = fraud signal',
)
try:
    dpan_fv = fs.register_feature_view(dpan_fv, version='V1', block=True)
    print('  Registered: FRAUD_DPAN_VELOCITY')
except Exception as e:
    if 'already exists' in str(e).lower():
        dpan_fv = fs.get_feature_view('FRAUD_DPAN_VELOCITY', 'V1')
        print('  Already registered: FRAUD_DPAN_VELOCITY')
    else:
        raise

# ── IP Address velocity ───────────────────────────────────────────────────────
# IP-level features detect botnet traffic and proxy/VPN fraud rings.
# Key signal: IP_DISTINCT_CUST_L1H > threshold means many different customers
# are transacting from the same IP in a short window — typical of a bot farm or NAT proxy.
ip_fv = FeatureView(
    name='FRAUD_IP_VELOCITY',
    entities=[ip_entity],
    stream_config=stream_cfg, timestamp_col='EVENT_TS',
    refresh_freq='1 minute', feature_granularity='1 minute',
    features=[
        Feature.count( 'AMOUNT_USD',   '1h' ).alias('IP_TXN_NUM_L1H'),
        Feature.count( 'AMOUNT_USD',   '6h' ).alias('IP_TXN_NUM_L6H'),
        Feature.count( 'AMOUNT_USD',   '24h').alias('IP_TXN_NUM_L24H'),
        Feature.count( 'AMOUNT_USD',   '48h').alias('IP_TXN_NUM_L48H'),
        Feature.count( 'AMOUNT_USD',   '7d' ).alias('IP_TXN_NUM_L1WK'),
        Feature.sum(   'AMOUNT_USD',   '24h').alias('IP_TXN_AMT_L24H'),
        Feature.sum(   'AMOUNT_USD',   '7d' ).alias('IP_TXN_AMT_L1WK'),
        Feature.approx_count_distinct('CUSTOMER_ID', '1h' ).alias('IP_DISTINCT_CUST_L1H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '24h').alias('IP_DISTINCT_CUST_L24H'),
        Feature.approx_count_distinct('CUSTOMER_ID', '7d' ).alias('IP_DISTINCT_CUST_L1WK'),
    ],
    online_config=online_cfg,
    feature_aggregation_method=FeatureAggregationMethod.CONTINUOUS,
    desc='IP velocity — shared IP = bot farm signal',
)
try:
    ip_fv = fs.register_feature_view(ip_fv, version='V1', block=True)
    print('  Registered: FRAUD_IP_VELOCITY')
except Exception as e:
    if 'already exists' in str(e).lower():
        ip_fv = fs.get_feature_view('FRAUD_IP_VELOCITY', 'V1')
        print('  Already registered: FRAUD_IP_VELOCITY')
    else:
        raise
print('\nAll 4 entity stream feature views registered.')

## 2.5 Customer Profile Feature View (Batch, Daily Refresh)

Static lifetime features that change slowly. Served from a DT-backed batch online feature view refreshed daily.

In [ ]:
# The customer profile feature view is a BATCH feature view (not CONTINUOUS/stream-based).
# It reads from a pre-aggregated table (CUSTOMER_PROFILE) that is refreshed daily.
# Batch feature views are appropriate for slowly-changing attributes like:
#   - Lifetime transaction count and total spend
#   - Account age in days
#   - Average transaction amount (30-day rolling)
# These features don't need sub-second freshness — daily refresh is sufficient.

# CUSTOMER_PROFILE doesn't exist as a table — build it by joining DIM_CUSTOMERS
# (static attributes) with aggregates from FRAUD_TRANSACTIONS.
session.sql('''
    CREATE TABLE IF NOT EXISTS FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE AS
    SELECT
        c.CUSTOMER_ID,
        c.CUSTOMER_AGE,
        c.DAYS_SINCE_REGISTRATION,
        c.IS_HIGH_RISK,
        COALESCE(t.LIFETIME_TXN_COUNT, 0)    AS LIFETIME_TXN_COUNT,
        COALESCE(t.LIFETIME_TXN_TOTAL, 0.0)  AS LIFETIME_TXN_TOTAL,
        COALESCE(t.AVG_TXN_AMOUNT_30D, 0.0)  AS AVG_TXN_AMOUNT_30D
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.DIM_CUSTOMERS c
    LEFT JOIN (
        SELECT
            CUSTOMER_ID,
            COUNT(*)                          AS LIFETIME_TXN_COUNT,
            SUM(PURCHASE_AMOUNT)              AS LIFETIME_TXN_TOTAL,
            AVG(CASE WHEN TRANSACTION_TS >= DATEADD(DAY, -30, CURRENT_TIMESTAMP())
                     THEN PURCHASE_AMOUNT END) AS AVG_TXN_AMOUNT_30D
        FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS
        GROUP BY CUSTOMER_ID
    ) t ON c.CUSTOMER_ID = t.CUSTOMER_ID
''').collect()
print('Created: FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE')

profile_src = session.table('FRAUD_DEMO_DEV.TRANSACTIONS.CUSTOMER_PROFILE')

profile_fv = FeatureView(
    name='FRAUD_CUSTOMER_PROFILE',
    entities=[customer_entity],
    feature_df=profile_src,
    refresh_freq='1 day',
    online_config=OnlineConfig(enable=True, target_lag='1h', store_type=OnlineStoreType.POSTGRES),
    desc='Customer profile: lifetime stats, account age. Batch online FV, daily refresh.',
)
try:
    profile_fv = fs.register_feature_view(profile_fv, version='V1', block=True)
    print('Registered: FRAUD_CUSTOMER_PROFILE (batch, daily refresh)')
except Exception as e:
    if 'already exists' in str(e).lower():
        profile_fv = fs.get_feature_view('FRAUD_CUSTOMER_PROFILE', 'V1')
        print('Already registered: FRAUD_CUSTOMER_PROFILE')
    else:
        raise

## 2.6 Register Feature Group (Single-Call Scoring)

`FeatureGroup` bundles all 5 feature views into a single `read_feature_group()` call.
This replaces 4-5 concurrent REST calls with **one round-trip** to the online service.

All source FVs must use `store_type=OnlineStoreType.POSTGRES` — verified above.

In [ ]:
# FeatureGroup bundles all 5 feature views for single-call scoring.
# 'a single read_feature_group() call returns features from all sources in one round-trip'
# This reduces 5 concurrent REST calls -> 1 call on the hot path.
#
# Use fs.get_feature_view() to retrieve the *registered* versions from the server.
# FeatureGroup requires registered FV objects, not local unregistered ones.
# This also makes the cell idempotent: works whether FVs were registered this session or earlier.
fraud_fg = FeatureGroup(
    name='FRAUD_SCORING_FG',
    features=[
        fs.get_feature_view('FRAUD_CUSTOMER_VELOCITY', 'V1'),
        fs.get_feature_view('FRAUD_MERCHANT_VELOCITY', 'V1'),
        fs.get_feature_view('FRAUD_DPAN_VELOCITY',     'V1'),
        fs.get_feature_view('FRAUD_IP_VELOCITY',       'V1'),
        fs.get_feature_view('FRAUD_CUSTOMER_PROFILE',  'V1'),
    ],
    auto_prefix=False,
    desc='All fraud scoring features: 4 velocity views (CONTINUOUS, <2s) + customer profile',
)

try:
    registered_fg = fs.register_feature_group(fraud_fg, version='V1')
    print(f'Registered FeatureGroup: {registered_fg.name} V1')
except Exception as e:
    if 'already exists' in str(e).lower():
        registered_fg = fs.get_feature_group('FRAUD_SCORING_FG', 'V1')
        print(f'FeatureGroup already exists: {registered_fg.name} V1')
    else:
        raise

print()
print('Scoring path: fs.read_feature_group(fraud_fg, keys=[[cust_id, merch_id, dpan, ip]])')
print('  -> ONE round-trip -> all velocity + profile features')
print('  -> XGBoost.predict() (~5ms)')


## 2.6 Freshness & Latency Benchmarks (OPTIONAL)

**These cells are optional — skip to nb03 if the OFS setup above completed without errors.**

Requirements to run:
- A PAT set in the PAT setup cell (section 2.0)
- No network policy blocking the Notebooks compute IP on your account

The definitive freshness proof and latency numbers are in **nb06_latency_proof.ipynb**,
which calls the SPCS scoring service using internal OFS URLs — no PAT or network policy required.


In [ ]:
import os
if not os.environ.get('SNOWFLAKE_PAT'):
    print('SKIPPED: SNOWFLAKE_PAT not set.')
    print('Paste your PAT in the PAT setup cell (section 2.0) and re-run.')
    raise SystemExit('Set SNOWFLAKE_PAT to run this benchmark')

# Freshness benchmark: measures how quickly an ingested event propagates through
# the CONTINUOUS pipeline to all 4 feature views.
#
# Uses the Python SDK (fs.stream_ingest + fs.read_feature_view) rather than direct
# REST calls. The OFS public HTTPS endpoint requires a PAT; the SDK authenticates
# through the session automatically, no PAT needed.
#
# True REST latency from SPCS is shown in nb06_latency_proof.ipynb.

import time, random
from datetime import datetime, timezone

print('='*60)
print('FRESHNESS BENCHMARK')
print('='*60)
print('''
Ingest 1 transaction via the OFS SDK (stream_ingest).
Poll all 4 CONTINUOUS feature views until count increments 0 -> 1.
Latency = time from ingest call to first successful query showing the update.
Target: < 2 seconds.
''')

# Retrieve registered FV objects — needed for read_feature_view
cust_fv_reg  = fs.get_feature_view('FRAUD_CUSTOMER_VELOCITY', 'V1')
merch_fv_reg = fs.get_feature_view('FRAUD_MERCHANT_VELOCITY', 'V1')
dpan_fv_reg  = fs.get_feature_view('FRAUD_DPAN_VELOCITY',     'V1')
ip_fv_reg    = fs.get_feature_view('FRAUD_IP_VELOCITY',       'V1')

FV_INFO = [
    (cust_fv_reg,  'CUSTOMER_ID', 'PURCHASES_NUM_L1H'),
    (merch_fv_reg, 'MERCHANT_ID', 'MERCHANT_TXN_NUM_L1H'),
    (dpan_fv_reg,  'WALLET_DPAN', 'DPAN_TXN_NUM_L1H'),
    (ip_fv_reg,    'IP_ADDRESS',  'IP_TXN_NUM_L1H'),
]

def sdk_query_count(fv, key_col, key_val, count_feature):
    """Read a single count feature via the SDK. Returns value or None."""
    try:
        rows = fs.read_feature_view(
            fv, keys=[[key_val]], store_type='online'
        ).to_pandas()
        if rows.empty:
            return None
        val = rows.iloc[0].get(count_feature)
        return val if val is not None else 0
    except Exception:
        return None

# Brand-new keys — guaranteed 0 prior history
test_cust  = f'BENCH_CUST_{random.randint(900000, 999999)}'
test_merch = f'BENCH_MERCH_{random.randint(900000, 999999)}'
test_dpan  = f'BENCH_DPAN_{random.randint(900000, 999999)}'
test_ip    = f'BENCH_IP_{random.randint(900000, 999999)}'
entity_keys = {
    'CUSTOMER_ID': test_cust, 'MERCHANT_ID': test_merch,
    'WALLET_DPAN': test_dpan, 'IP_ADDRESS':  test_ip,
}
print(f'Test keys: {test_cust} | {test_merch} | {test_dpan} | {test_ip}')

# Step 1: Baselines
print('\n--- Baselines (expect 0 for unseen keys) ---')
for fv, key_col, feat in FV_INFO:
    val = sdk_query_count(fv, key_col, entity_keys[key_col], feat)
    print(f'  {fv.name}: {feat} = {val}')

# Step 2: Ingest via SDK (stream_ingest — same REST call internally, correct auth)
print('\n--- Ingest ---')
ingest_ts = datetime.now(timezone.utc)
accepted = fs.stream_ingest(txn_stream, [{
    'CUSTOMER_ID': test_cust,  'MERCHANT_ID': test_merch,
    'WALLET_DPAN': test_dpan,  'IP_ADDRESS':  test_ip,
    'AMOUNT_USD':  round(random.uniform(10, 500), 2),
    'IS_GBR':      1.0,
    'EVENT_TS':    ingest_ts.strftime('%Y-%m-%dT%H:%M:%SZ'),
}])
print(f'  Ingested {accepted} record(s)')
t_start = time.time()

# Step 3: Poll all 4 FVs until count > 0
print('\n--- Polling (250ms intervals, 30s timeout) ---')
latencies = {}
pending   = {fv.name: (fv, key_col, feat) for fv, key_col, feat in FV_INFO}

for _ in range(120):  # 120 * 250ms = 30s
    time.sleep(0.25)
    for fv_name in list(pending):
        fv, key_col, feat = pending[fv_name]
        val = sdk_query_count(fv, key_col, entity_keys[key_col], feat)
        if val is not None and val > 0:
            ms = (time.time() - t_start) * 1000
            latencies[fv_name] = ms
            del pending[fv_name]
            print(f'  {fv_name}: {ms:.0f}ms  ({feat} = {val})')
    if not pending:
        break

# Step 4: Summary
print('\n--- Results ---')
if pending:
    print(f'  TIMEOUT (>30s): {list(pending)}')
if latencies:
    worst = max(latencies.values())
    print(f'  Fastest : {min(latencies.values()):.0f}ms')
    print(f'  Slowest : {worst:.0f}ms  (time-to-score = slowest of 4)')
    verdict = 'PASS' if worst < 2000 else 'REVIEW'
    print(f'  Target < 2000ms : {verdict}')


## 2.7 Query Latency Benchmark (OPTIONAL)

See note in 2.6 — skip to nb03 if setup is complete.


In [ ]:
import os
if not os.environ.get('SNOWFLAKE_PAT'):
    print('SKIPPED: SNOWFLAKE_PAT not set.')
    print('Paste your PAT in the PAT setup cell (section 2.0) and re-run.')
    raise SystemExit('Set SNOWFLAKE_PAT to run this benchmark')

# Query latency benchmark using the SDK (no PAT required).
# Measures the round-trip time for fs.read_feature_view() calls from Notebooks.
#
# Note: SDK calls go through Snowflake's internal auth path; the latency here
# includes Python overhead and Snowflake SQL compilation not present in raw REST.
# For true REST latency from SPCS (lowest latency path), see nb06_latency_proof.ipynb.

import concurrent.futures, numpy as np, time, random

print('='*60)
print('QUERY LATENCY BENCHMARK  (SDK, from Notebooks)')
print('='*60)
print('Measures fs.read_feature_view() round-trip latency.')
print('For production-equivalent SPCS internal REST latency, see nb06.\n')

N_WARMUP, N_REQUESTS = 5, 50

# Sample real entity keys
samples = session.sql(
    'SELECT CUSTOMER_ID, MERCHANT_ID, WALLET_DPAN, IP_ADDRESS '
    'FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS SAMPLE (100 ROWS)'
).collect()
print(f'Sampled {len(samples)} entity keys\n')

# Get registered FVs (may already be set from freshness cell above)
try:
    cust_fv_reg
except NameError:
    cust_fv_reg  = fs.get_feature_view('FRAUD_CUSTOMER_VELOCITY', 'V1')
    merch_fv_reg = fs.get_feature_view('FRAUD_MERCHANT_VELOCITY', 'V1')
    dpan_fv_reg  = fs.get_feature_view('FRAUD_DPAN_VELOCITY',     'V1')
    ip_fv_reg    = fs.get_feature_view('FRAUD_IP_VELOCITY',       'V1')

def single_fv_lookup(fv, key_val):
    t0 = time.perf_counter()
    fs.read_feature_view(fv, keys=[[key_val]], store_type='online').collect()
    return (time.perf_counter() - t0) * 1000

def all_4_concurrent(row):
    """4 FV lookups in parallel — wall-clock = max(4)."""
    t0 = time.perf_counter()
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as ex:
        futs = [
            ex.submit(single_fv_lookup, cust_fv_reg,  row['CUSTOMER_ID']),
            ex.submit(single_fv_lookup, merch_fv_reg, row['MERCHANT_ID']),
            ex.submit(single_fv_lookup, dpan_fv_reg,  row['WALLET_DPAN']),
            ex.submit(single_fv_lookup, ip_fv_reg,    row['IP_ADDRESS']),
        ]
        [f.result() for f in concurrent.futures.as_completed(futs)]
    return (time.perf_counter() - t0) * 1000

# Warmup
print(f'Warming up ({N_WARMUP} requests)...')
for row in samples[:N_WARMUP]:
    all_4_concurrent(row)
print('  Done.\n')

# Measured
print(f'Running {N_REQUESTS} measured requests...')
latencies = []
for i in range(N_REQUESTS):
    latencies.append(all_4_concurrent(samples[i % len(samples)]))
    if (i + 1) % 10 == 0:
        print(f'  {i+1}/{N_REQUESTS}...')

arr = np.array(latencies)
print()
print(f'4-entity concurrent SDK lookup  (n={N_REQUESTS})')
print(f'  p50: {np.percentile(arr,50):.1f}ms')
print(f'  p95: {np.percentile(arr,95):.1f}ms')
print(f'  p99: {np.percentile(arr,99):.1f}ms')
print(f'\nNote: SDK latency from Notebooks is higher than SPCS internal REST.')
print(f'See nb06_latency_proof.ipynb for production-equivalent numbers.')


---
## Summary

| Component | Status | Freshness | Method |
|---|---|---|---|
| FRAUD_CUSTOMER_VELOCITY | Registered | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_MERCHANT_VELOCITY | Registered | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_DPAN_VELOCITY | Registered | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_IP_VELOCITY | Registered | < 2 seconds | CONTINUOUS stream aggregation |
| FRAUD_CUSTOMER_PROFILE | Registered | Daily | Batch online FV |
| FRAUD_SCORING_FG | Registered | N/A | Feature Group (all 5 FVs, one round-trip) |

The freshness and latency benchmarks (sections 2.6–2.7) require a PAT and
a permissive network policy. **Skip them if they fail — the demo numbers come from nb06.**

**Next: nb03_training.ipynb** — train XGBoost from the transactions table.
